In [1]:
pip install yfinance feedparser requests


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import feedparser
import json
import time
from datetime import datetime, timedelta
from bs4 import BeautifulSoup

# Define the 90-day historical window
DAYS_HISTORY = 90
target_date = datetime.now() - timedelta(days=DAYS_HISTORY)
unix_target_date = int(target_date.timestamp())

def scrape_hackernews_historical(query="nvidia"):
    print(f"Scraping HackerNews: Past {DAYS_HISTORY} days")
    # Using Algolia's public API to bypass standard HN limitations
    url = f"http://hn.algolia.com/api/v1/search?query={query}&numericFilters=created_at_i>{unix_target_date}&hitsPerPage=100"
    
    # Adding a generic User-Agent to prevent basic blocking
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}
    response = requests.get(url, headers=headers)
    
    data = []
    if response.status_code == 200:
        hits = response.json().get('hits', [])
        for hit in hits:
            # We only want posts with actual discussion/comments
            if hit.get('num_comments', 0) > 2:
                data.append({
                    "source": "HackerNews",
                    "url": hit.get('url', f"https://news.ycombinator.com/item?id={hit.get('objectID')}"),
                    "title": hit.get('title', ''),
                    "summary": hit.get('story_text', '') or hit.get('title', ''),
                    # Force strict ISO date formatting for the trend graph later
                    "date": datetime.fromtimestamp(hit.get('created_at_i')).isoformat()
                })
    print(f"Retrieved {len(data)} HackerNews threads.")
    return data

def scrape_market_news_historical(query="NVIDIA"):
    print(f"Scraping Market News: Past {DAYS_HISTORY} days")
    # Bypassing yfinance limits using a parameterized news query
    rss_url = f"https://news.google.com/rss/search?q={query}+when:{DAYS_HISTORY}d&hl=en-US&gl=US&ceid=US:en"
    
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}
    response = requests.get(rss_url, headers=headers)
    feed = feedparser.parse(response.content)
    
    data = []
    for entry in feed.entries:
        try:
            # Convert messy RSS dates into clean ISO format
            parsed_date = datetime(*entry.published_parsed[:6]).isoformat()
        except:
            parsed_date = datetime.now().isoformat()
            
        data.append({
            "source": "Market News",
            "url": entry.link,
            "title": entry.title,
            "summary": entry.title, # Fallback, as RSS summaries are often just HTML links
            "date": parsed_date
        })
    print(f"Retrieved {len(data)} market news articles.")
    return data

def scrape_nvidia_rss():
    print("Scraping NVIDIA Official Newsroom")
    # Corporate firewalls block Python based on TLS fingerprinting.
    # We bypass this by outsourcing the scrape to Google News, filtering strictly for official NVIDIA domains.
    query = "site:nvidianews.nvidia.com"
    rss_url = f"https://news.google.com/rss/search?q={query}+when:{DAYS_HISTORY}d&hl=en-US&gl=US&ceid=US:en"
    
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}
    
    try:
        response = requests.get(rss_url, headers=headers, timeout=10)
        feed = feedparser.parse(response.content)
    except Exception as e:
        print(f"Failed to fetch NVIDIA official news via proxy: {e}")
        return []
    
    data = []
    for entry in feed.entries:
        try:
            parsed_date = datetime(*entry.published_parsed[:6]).isoformat()
        except:
            parsed_date = datetime.now().isoformat()
            
        data.append({
            "source": "NVIDIA Official",
            "url": entry.link,
            "title": entry.title,
            "summary": entry.title, 
            "date": parsed_date
        })
    print(f"Retrieved {len(data)} official press releases.")
    return data

# Execution Flow
if __name__ == "__main__":
    print(f"Initializing Historical Intelligence Sweep: {DAYS_HISTORY} Day Window")
    print("="*60)

    hn_data = scrape_hackernews_historical()
    market_data = scrape_market_news_historical()
    nvidia_data = scrape_nvidia_rss()

    # Merge and export
    all_raw_data = hn_data + market_data + nvidia_data
    output_file = "nvidia_raw_intelligence.jsonl"

    with open(output_file, "w", encoding="utf-8") as f:
        for item in all_raw_data:
            f.write(json.dumps(item) + "\n")

    print("="*60)
    print(f"Data collection complete. {len(all_raw_data)} total records saved to {output_file}.")

Initializing Historical Intelligence Sweep: 90 Day Window
Scraping HackerNews: Past 90 days...
Retrieved 35 HackerNews threads.
Scraping Market News: Past 90 days...
Retrieved 100 market news articles.
Scraping NVIDIA Official Newsroom...
Retrieved 47 official press releases.
Data collection complete. 182 total records saved to nvidia_raw_intelligence.jsonl.
